# IEMOCAP Preprocessing

For each utterance, extract:
- **audio/** — per-utterance `.wav` (copied from `sentences/wav/`)
- **video/** — per-utterance `.mp4` (speaker-cropped, audio stripped, extracted from dialog AVI)
- **text/** — per-utterance `.txt` (transcription)
- **labels.csv** — unified label manifest

IEMOCAP dialog videos are split-screen (two speakers side by side). This notebook crops to the
correct speaker half using the filename/utterance-ID convention from `video_processor.ipynb`:
- `Ses{NN}F_...` → Female speaker is on the LEFT side of the frame
- `Ses{NN}M_...` → Male speaker is on the LEFT side of the frame
- Utterance speaker determined by last segment of utterance ID: `_F{NNN}` → Female, `_M{NNN}` → Male

Output: `/mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/`

In [1]:
import os
import re
import shutil
import subprocess
import csv
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm

In [ ]:
BASE         = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
IEMOCAP_ROOT = BASE / "Dataset/IEMOCAP_full_release"
OUT_ROOT     = BASE / "Dataset/Processed/IEMOCAP"

AUDIO_DIR = OUT_ROOT / "audio"
VIDEO_DIR = OUT_ROOT / "video"
TEXT_DIR  = OUT_ROOT / "text"

for d in [AUDIO_DIR, VIDEO_DIR, TEXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SESSIONS = [f"Session{i}" for i in range(1, 6)]

## 1. Helper Functions

In [3]:
# ── utterance ID helpers ─────────────────────────────────────────────────────

def get_speaker(utt_id: str) -> str:
    """Return 'F' or 'M' — the speaker of this utterance."""
    # utt_id format: Ses01F_impro01_F000  →  last segment suffix before digits
    suffix = utt_id.rsplit('_', 1)[-1]  # e.g. 'F000'
    return suffix[0]  # 'F' or 'M'

def get_session(utt_id: str) -> str:
    return f"Session{int(utt_id[3:5])}"

def get_dialog(utt_id: str) -> str:
    # Ses01F_impro01_F000  →  Ses01F_impro01
    return utt_id.rsplit('_', 1)[0]

def get_left_speaker_gender(dialog_name: str) -> str:
    """
    The character at index 5 of the dialog/session filename encodes which gender
    is on the LEFT side of the split-screen frame.
    e.g. 'Ses01F_impro01' → index 5 = 'F' → Female is on the left
         'Ses01M_impro01' → index 5 = 'M' → Male is on the left
    """
    return dialog_name[5]  # 'F' or 'M'

def get_crop_params(dialog_name: str, utt_speaker: str) -> tuple[str, str]:
    """
    Returns (x_offset, audio_channel) for FFmpeg.
    - x_offset '0'     → crop left half   (audio channel FL)
    - x_offset 'iw/2'  → crop right half  (audio channel FR)
    """
    left_gender = get_left_speaker_gender(dialog_name)
    if utt_speaker == left_gender:
        return '0', 'FL'       # speaker is on the left
    else:
        return 'iw/2', 'FR'    # speaker is on the right

# Sanity check
x, ch = get_crop_params('Ses01F_impro01', 'F')  # Female left, female speaker → left
print(f"Female speaker in Ses01F session → x_offset={x}, channel={ch}")  # expect 0, FL
x, ch = get_crop_params('Ses01F_impro01', 'M')  # Female left, male speaker → right
print(f"Male speaker in Ses01F session   → x_offset={x}, channel={ch}")  # expect iw/2, FR

Female speaker in Ses01F session → x_offset=0, channel=FL
Male speaker in Ses01F session   → x_offset=iw/2, channel=FR


## 2. Parse Labels (EmoEvaluation)

In [4]:
# [6.2901 - 8.2357]\tSes01F_impro01_F000\tneu\t[2.5000, 2.5000, 2.5000]
_EMO_LINE = re.compile(
    r'\[([\d.]+)\s*-\s*([\d.]+)\]\s+(\S+)\s+(\S+)\s+\[([\d.]+),\s*([\d.]+),\s*([\d.]+)\]'
)

def parse_emo_evaluation(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding='latin-1').splitlines():
        m = _EMO_LINE.search(line)
        if m:
            start, end, utt_id, emotion, val, aro, dom = m.groups()
            rows.append({
                "utt_id":     utt_id,
                "emotion":    emotion,
                "start_time": float(start),
                "end_time":   float(end),
                "valence":    float(val),
                "arousal":    float(aro),
                "dominance":  float(dom),
            })
    return rows

# Test
test_rows = parse_emo_evaluation(
    IEMOCAP_ROOT / "Session1/dialog/EmoEvaluation/Ses01F_impro01.txt"
)
print(f"Sample dialog: {len(test_rows)} utterances")
print(test_rows[0])

Sample dialog: 30 utterances
{'utt_id': 'Ses01F_impro01_F000', 'emotion': 'neu', 'start_time': 6.2901, 'end_time': 8.2357, 'valence': 2.5, 'arousal': 2.5, 'dominance': 2.5}


## 3. Parse Transcriptions

In [5]:
# Format: Ses01F_impro01_F000 [006.2901-008.2357]: Excuse me.
_TRANS_LINE = re.compile(r'^(\S+)\s+\[[^\]]+\]:\s*(.+)$')

def parse_transcriptions(path: Path) -> dict[str, str]:
    result = {}
    for line in path.read_text(encoding='latin-1').splitlines():
        m = _TRANS_LINE.match(line.strip())
        if m:
            utt_id, text = m.groups()
            result[utt_id] = text.strip()
    return result

# Test
test_trans = parse_transcriptions(
    IEMOCAP_ROOT / "Session1/dialog/transcriptions/Ses01F_impro01.txt"
)
print(f"Sample dialog: {len(test_trans)} transcription entries")
print(next(iter(test_trans.items())))

Sample dialog: 30 transcription entries
('Ses01F_impro01_F000', 'Excuse me.')


## 4. Collect All Records

In [6]:
_DIALOG_EMO_FILE = re.compile(r"^Ses\d{2}[FM]_(?:impro\d{2}[a-z]?|script\d{2}_\d[a-z]?)\.txt$")

all_records = []

for session in SESSIONS:
    emo_dir   = IEMOCAP_ROOT / session / "dialog" / "EmoEvaluation"
    trans_dir = IEMOCAP_ROOT / session / "dialog" / "transcriptions"
    wav_root  = IEMOCAP_ROOT / session / "sentences" / "wav"
    avi_dir   = IEMOCAP_ROOT / session / "dialog" / "avi" / "DivX"

    emo_files = sorted(p for p in emo_dir.glob("*.txt") if _DIALOG_EMO_FILE.match(p.name))

    for emo_file in emo_files:
        dialog = emo_file.stem  # e.g. Ses01F_impro01
        trans_file = trans_dir / f"{dialog}.txt"
        dialog_avi = avi_dir / f"{dialog}.avi"

        emo_rows  = parse_emo_evaluation(emo_file)
        trans_map = parse_transcriptions(trans_file) if trans_file.exists() else {}

        for row in emo_rows:
            utt_id  = row["utt_id"]
            speaker = get_speaker(utt_id)
            x_off, audio_ch = get_crop_params(dialog, speaker)
            src_wav = wav_root / dialog / f"{utt_id}.wav"

            all_records.append({
                **row,
                "session":      session,
                "dialog":       dialog,
                "speaker":      speaker,
                "transcription": trans_map.get(utt_id, ""),
                "src_wav":      str(src_wav),
                "dialog_avi":   str(dialog_avi),
                "x_offset":     x_off,
                "audio_ch":     audio_ch,
                "has_wav":      src_wav.exists(),
                "has_avi":      dialog_avi.exists(),
            })

print(f"Total utterances: {len(all_records)}")
missing_wav = sum(1 for r in all_records if not r["has_wav"])
missing_avi = sum(1 for r in all_records if not r["has_avi"])
print(f"Missing WAV: {missing_wav} | Missing dialog AVI: {missing_avi}")

Total utterances: 10039
Missing WAV: 0 | Missing dialog AVI: 0


## 5. Copy Audio Files

IEMOCAP already provides per-utterance `.wav` files in `sentences/wav/`. Just copy them.

In [7]:
copied = 0
skipped = 0
audio_errors = []

for rec in tqdm(all_records, desc="Copying audio"):
    dst = AUDIO_DIR / f"{rec['utt_id']}.wav"
    if dst.exists():
        skipped += 1
        continue
    if rec["has_wav"]:
        shutil.copy2(rec["src_wav"], dst)
        copied += 1
    else:
        audio_errors.append(rec["utt_id"])

print(f"Copied: {copied} | Skipped (already exists): {skipped} | Errors: {len(audio_errors)}")
if audio_errors:
    print("Missing WAV utterances:", audio_errors[:10])

Copying audio: 100%|██████████| 10039/10039 [00:09<00:00, 1088.85it/s]

Copied: 10039 | Skipped (already exists): 0 | Errors: 0


## 6. Extract Speaker-Cropped Video

Each dialog AVI has two speakers side by side. For each utterance:
1. Seek to utterance start time (`-ss`)
2. Read only utterance duration (`-t`)
3. Crop to the speaker's half of the frame (`-vf crop=iw/2:ih:{x_offset}:0`)
4. Drop audio stream (`-an`)

In [8]:
def extract_video_only(rec: dict) -> tuple[str, str]:
    """Returns (utt_id, 'ok'|'skip'|'error: msg')."""
    utt_id = rec["utt_id"]
    dst = VIDEO_DIR / f"{utt_id}.mp4"
    if dst.exists():
        return utt_id, "skip"
    if not rec["has_avi"]:
        return utt_id, "error: missing dialog AVI"

    start    = rec["start_time"]
    duration = rec["end_time"] - rec["start_time"]
    x_off    = rec["x_offset"]  # '0' or 'iw/2'

    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-ss", str(start),          # seek to start (fast seek before -i)
        "-i", rec["dialog_avi"],
        "-t", str(duration),         # clip duration
        "-vf", f"crop=iw/2:ih:{x_off}:0",  # crop speaker half
        "-an",                        # no audio
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "18",
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return utt_id, f"error: {result.stderr.strip()[:200]}"
    return utt_id, "ok"

video_results = Parallel(n_jobs=-1)(
    delayed(extract_video_only)(rec)
    for rec in tqdm(all_records, desc="Extracting video")
)

ok   = sum(1 for _, s in video_results if s == "ok")
skip = sum(1 for _, s in video_results if s == "skip")
errs = [(u, s) for u, s in video_results if s.startswith("error")]
print(f"OK: {ok} | Skipped: {skip} | Errors: {len(errs)}")
if errs:
    for u, e in errs[:10]:
        print(f"  {u}: {e}")

Extracting video: 100%|██████████| 10039/10039 [14:15<00:00, 11.73it/s]


OK: 10039 | Skipped: 0 | Errors: 0


## 7. Write Text Files

In [9]:
written = 0
skipped_text = 0
empty_trans  = 0

for rec in tqdm(all_records, desc="Writing text"):
    dst = TEXT_DIR / f"{rec['utt_id']}.txt"
    if dst.exists():
        skipped_text += 1
        continue
    text = rec["transcription"]
    if not text:
        empty_trans += 1
    dst.write_text(text, encoding="utf-8")
    written += 1

print(f"Written: {written} | Skipped: {skipped_text} | Empty transcriptions: {empty_trans}")

Writing text: 100%|██████████| 10039/10039 [00:00<00:00, 12683.59it/s]

Written: 10039 | Skipped: 0 | Empty transcriptions: 0


## 8. Write labels.csv

In [10]:
labels_path = OUT_ROOT / "labels.csv"
fieldnames = [
    "utt_id", "session", "dialog", "speaker",
    "emotion", "valence", "arousal", "dominance",
    "start_time", "end_time", "transcription",
    "audio_path", "video_path", "text_path"
]

with open(labels_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for rec in all_records:
        utt_id = rec["utt_id"]
        writer.writerow({
            "utt_id":        utt_id,
            "session":       rec["session"],
            "dialog":        rec["dialog"],
            "speaker":       rec["speaker"],
            "emotion":       rec["emotion"],
            "valence":       rec["valence"],
            "arousal":       rec["arousal"],
            "dominance":     rec["dominance"],
            "start_time":    rec["start_time"],
            "end_time":      rec["end_time"],
            "transcription": rec["transcription"],
            "audio_path":    f"audio/{utt_id}.wav",
            "video_path":    f"video/{utt_id}.mp4",
            "text_path":     f"text/{utt_id}.txt",
        })

print(f"labels.csv written: {labels_path}")
print(f"Total rows: {len(all_records)}")

labels.csv written: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/labels.csv
Total rows: 10039


## 9. Verification

In [11]:
import pandas as pd

df = pd.read_csv(labels_path)
print(f"Labels CSV rows: {len(df)}")

print(f"\nEmotion distribution:")
print(df["emotion"].value_counts())

print(f"\nSpeaker distribution:")
print(df["speaker"].value_counts())

audio_count = len(list(AUDIO_DIR.glob("*.wav")))
video_count = len(list(VIDEO_DIR.glob("*.mp4")))
text_count  = len(list(TEXT_DIR.glob("*.txt")))
print(f"\naudio/: {audio_count} .wav files")
print(f"video/: {video_count} .mp4 files")
print(f"text/:  {text_count} .txt files")

# Spot check first utterance
sample = df.iloc[0]
print(f"\nSpot check — {sample.utt_id}:")
print(f"  emotion: {sample.emotion}")
print(f"  text:    {sample.transcription}")
print(f"  audio exists: {(AUDIO_DIR / sample.utt_id).with_suffix('.wav').exists()}")
print(f"  video exists: {(VIDEO_DIR / sample.utt_id).with_suffix('.mp4').exists()}")
print(f"  text  exists: {(TEXT_DIR  / sample.utt_id).with_suffix('.txt').exists()}")

# Verify no audio stream in a sample video
sample_video = VIDEO_DIR / f"{sample.utt_id}.mp4"
if sample_video.exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_streams",
         "-select_streams", "a", str(sample_video)],
        capture_output=True, text=True
    )
    has_audio = "codec_type=audio" in probe.stdout
    print(f"  video has audio stream: {has_audio}  (should be False)")

Labels CSV rows: 10039

Emotion distribution:
emotion
xxx    2507
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
sur     107
fea      40
oth       3
dis       2
Name: count, dtype: int64

Speaker distribution:
speaker
M    5239
F    4800
Name: count, dtype: int64

audio/: 10039 .wav files
video/: 10039 .mp4 files
text/:  10039 .txt files

Spot check — Ses01F_impro01_F000:
  emotion: neu
  text:    Excuse me.
  audio exists: True
  video exists: True
  text  exists: True
  video has audio stream: False  (should be False)
